# M04-L01 — Introdução a Redes Neurais e PyTorch
## Laboratório

**Antes de começar:**

1. Abra este notebook no Google Colab.
2. Vá em **Arquivo → Salvar uma cópia no Drive** antes de editar qualquer célula.
3. Execute todas as células **na ordem**, de cima para baixo, incluindo a retomada condensada do M03.
4. O PyTorch já vem instalado por padrão no Google Colab — a célula abaixo apenas confirma isso.
5. Este notebook roda inteiramente em **CPU**. Não é necessário ativar GPU nesta lição.
6. Ao final, você deve entregar este notebook com todas as células executadas e todas as respostas em texto preenchidas.


In [5]:
import torch
print("Versão do PyTorch:", torch.__version__)


Versão do PyTorch: 2.11.0+cpu


## 1. Retomada condensada do M03

Antes de conhecer PyTorch, vamos reconstruir rapidamente a tabela `df_features`, exatamente como ela ficou ao final da M03-L06: as mesmas três fotos do Iris, a mesma função `extrair_features_imagem` e a mesma semente `np.random.seed(42)`. Esta célula não repete a exploração completa já feita no M03 — apenas reconstrói o ponto de partida desta lição.


In [6]:
import numpy as np
import pandas as pd
import urllib.request
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt
import time

URLS_IRIS = {
    "iris_setosa.jpg": "https://upload.wikimedia.org/wikipedia/commons/5/56/Kosaciec_szczecinkowaty_Iris_setosa.jpg",
    "iris_versicolor.jpg": "https://upload.wikimedia.org/wikipedia/commons/4/41/Iris_versicolor_3.jpg",
    "iris_virginica.jpg": "https://upload.wikimedia.org/wikipedia/commons/9/9f/Iris_virginica.jpg",
}

for nome_arquivo, url in URLS_IRIS.items():
    req = urllib.request.Request(
        url,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    time.sleep(1)  # Evita muitas requisições em sequência

    with urllib.request.urlopen(req) as response:
        with open(nome_arquivo, "wb") as f:
            f.write(response.read())

imagens_iris = {
    nome_arquivo: Image.open(nome_arquivo).convert("RGB")
    for nome_arquivo in URLS_IRIS
}

print("Imagens carregadas:", list(imagens_iris.keys()))

Imagens carregadas: ['iris_setosa.jpg', 'iris_versicolor.jpg', 'iris_virginica.jpg']


In [7]:
def extrair_features_imagem(imagem_pil):
    """Reaproveitada sem alteração da M03-L05/M03-L06."""
    arr = np.array(imagem_pil).astype(np.float32) / 255.0
    media_r, media_g, media_b = arr[:, :, 0].mean(), arr[:, :, 1].mean(), arr[:, :, 2].mean()
    cinza = arr.mean(axis=2)
    std_cinza = cinza.std()
    prop_segmentada = (cinza > cinza.mean()).mean()
    return media_r, media_g, media_b, std_cinza, prop_segmentada

np.random.seed(42)

linhas = []
for classe, imagem in imagens_iris.items():
    n_amostras = np.random.randint(15, 31)
    for _ in range(n_amostras):
        base = np.array(extrair_features_imagem(imagem))
        ruido = np.random.normal(loc=0.0, scale=0.01, size=base.shape)
        media_r, media_g, media_b, std_cinza, prop_segmentada = base + ruido
        linhas.append({
            "media_r": media_r, "media_g": media_g, "media_b": media_b,
            "std_cinza": std_cinza, "prop_segmentada": prop_segmentada,
            "classe": classe,
        })

df_features = pd.DataFrame(linhas)
df_features.to_csv("df_features.csv", index=False)
df_features.head()


,media_r,media_g,media_b,std_cinza,prop_segmentada,classe
0,0.418250,0.463520,0.320472,0.180413,0.480365,iris_setosa.jpg
1,0.422511,0.438256,0.310805,0.170654,0.480242,iris_setosa.jpg
2,0.424551,0.456770,0.315956,0.162451,0.484216,iris_setosa.jpg
3,0.422578,0.460586,0.308054,0.168153,0.489187,iris_setosa.jpg
4,0.435096,0.457318,0.310482,0.185856,0.469266,iris_setosa.jpg


**IMPORTANTE:** Ao executar a célula de código abaixo, será pedido a permissão para o colab acessar o seu Google Drive. É necessário que você permita para ele salvar ou recuperar arquivos que poderão ser usados nesta ou em lições futuras.

In [8]:
import os
import shutil
from google.colab import drive

# 1. Conecta o Google Drive do aluno
drive.mount('/content/drive')

# 2. Defina os caminhos
nome_arquivo_csv = 'df_features.csv'
caminho_origem_no_colab = os.path.join('/content', nome_arquivo_csv)
pasta_destino_no_drive = '/content/drive/MyDrive/dados_modulo_04'
caminho_destino_arquivo_no_drive = os.path.join(pasta_destino_no_drive, nome_arquivo_csv)

# 3. Copia o arquivo para o Drive
if os.path.exists(caminho_origem_no_colab):

    os.makedirs(pasta_destino_no_drive, exist_ok=True)

    shutil.copy(caminho_origem_no_colab, caminho_destino_arquivo_no_drive)
    print(f"Sucesso! O arquivo '{nome_arquivo_csv}' foi salvo permanentemente no seu Google Drive em: {caminho_destino_arquivo_no_drive}")
else:
    print(f"Erro: O arquivo '{nome_arquivo_csv}' não foi encontrado na memória do Colab. Certifique-se de rodar os blocos anteriores.")

Mounted at /content/drive
Sucesso! O arquivo 'df_features.csv' foi salvo permanentemente no seu Google Drive em: /content/drive/MyDrive/dados_modulo_04/df_features.csv


In [9]:
df_features.info()
df_features["classe"].value_counts()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64 entries, 0 to 63
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   media_r          64 non-null     float64
 1   media_g          64 non-null     float64
 2   media_b          64 non-null     float64
 3   std_cinza        64 non-null     float64
 4   prop_segmentada  64 non-null     float64
 5   classe           64 non-null     object 
dtypes: float64(5), object(1)
memory usage: 3.1+ KB


,count
classe,
iris_virginica.jpg,23
iris_setosa.jpg,21
iris_versicolor.jpg,20


### Exercício 1 (texto)

Reexecute as duas células acima (`.info()` e `.value_counts()`) sobre o `df_features` reconstruído.

**Responda em texto:** a tabela permanece balanceada entre as três classes após a reconstrução? Justifique com os números que você observou.

*Sua resposta aqui:*


## 2. Da fórmula fixa ao aprendiz

Relembre como um classificador clássico da M03-L06 era treinado (código já liberado, sem novidade nesta lição):


In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X = df_features[["media_r", "media_g", "media_b", "std_cinza", "prop_segmentada"]]
y = df_features["classe"]

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

modelo_classico = LogisticRegression(max_iter=1000)
modelo_classico.fit(X_treino, y_treino)
print("Acurácia no teste (classificador clássico):", modelo_classico.score(X_teste, y_teste))


Acurácia no teste (classificador clássico): 1.0


A `LogisticRegression` calcula seus coeficientes de uma só vez a partir de `X_treino`/`y_treino`. A rede neural que vamos construir a seguir começa de pesos aleatórios — o ajuste desses pesos só vai acontecer na M04-L02. Por enquanto, vamos apenas **construir a estrutura** da rede.

## 3. Convertendo `df_features` em tensores

O PyTorch não trabalha diretamente com `pandas.DataFrame`. Precisamos converter as features em um tensor `X_tensor` e o rótulo (codificado em inteiros) em um tensor `y_tensor`.


In [11]:
import torch

mapa_classes = {"setosa": 0, "versicolor": 1, "virginica": 2}
y_codificado = df_features["classe"].map(mapa_classes).values

X_tensor = torch.tensor(X.values, dtype=torch.float32)
y_tensor = torch.tensor(y_codificado, dtype=torch.long)

print("X_tensor.shape:", X_tensor.shape, "| X_tensor.dtype:", X_tensor.dtype)
print("y_tensor.shape:", y_tensor.shape, "| y_tensor.dtype:", y_tensor.dtype)


X_tensor.shape: torch.Size([64, 5]) | X_tensor.dtype: torch.float32
y_tensor.shape: torch.Size([64]) | y_tensor.dtype: torch.int64


### Exercício 2 (modificação guiada)

Com base na célula acima, instancie a váriavel `X_tensor_float64`. A célula abaixo já compara, para você, o `.dtype` e o consumo de memória (em bytes) dos dois tensores lado a lado — você não precisa escrever essa comparação, apenas alterar o `dtype` indicado e executar.

**Responda em texto:** o que muda no `.dtype` e no consumo de memória entre `X_tensor` (float32) e `X_tensor_float64`? Com base nessa comparação direta, por que você acha que esta lição escolheu `torch.float32` como padrão, em vez de `torch.float64`?

*Sua resposta aqui:*


In [12]:
# Exercício 2 — modificação guiada

X_tensor = torch.tensor(X.values, dtype=torch.float32)
X_tensor_float64 = # apague esse comentário e faça a instanciação da variavel

memoria_float32_bytes = X_tensor.element_size() * X_tensor.nelement()
memoria_float64_bytes = X_tensor_float64.element_size() * X_tensor_float64.nelement()

print("X_tensor (original)     | dtype:", X_tensor.dtype, "| memória (bytes):", memoria_float32_bytes)
print("X_tensor_float64 (novo) | dtype:", X_tensor_float64.dtype, "| memória (bytes):", memoria_float64_bytes)

SyntaxError: invalid syntax (1743908590.py, line 4)

## 4. `Dataset` e `DataLoader`

Vamos organizar `X_tensor` e `y_tensor` em um `Dataset` do PyTorch e, em seguida, em um `DataLoader` que entrega as amostras em pequenos lotes.


In [ ]:
from torch.utils.data import Dataset, DataLoader

class IrisFeaturesDataset(Dataset):
    def __init__(self, X_tensor, y_tensor):
        self.X = X_tensor
        self.y = y_tensor

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = IrisFeaturesDataset(X_tensor, y_tensor)
loader = DataLoader(dataset, batch_size=8)

X_lote, y_lote = next(iter(loader))
print("Shape de um lote de X:", X_lote.shape)
print("Shape de um lote de y:", y_lote.shape)


### Exercício 3 (modificação guiada)

Copie a célula abaixo **duas vezes**, alterando **apenas** o necessário para obter o valor de `batch_size` do `DataLoader` em cada cópia: uma vez para `batch_size=8` e outra para `batch_size=16`. Em cada cópia, inspecione o `.shape` do lote de `X_lote4`, `X_lote8` e `X_lote16` e de `y_lote4`, `y_lote8` e `y_lote16` gerado.

**Responda em texto:** compare os três casos — o `DataLoader` original com `batch_size=4` e os dois que você acabou de criar com `batch_size=8` e `batch_size=16`. O que muda e o que permanece igual no `.shape` de `X_lote` entre os três? Qual dimensão do lote é diretamente controlada por `batch_size`, e qual dimensão nunca muda, não importa o valor escolhido?

*Sua resposta aqui:*


In [ ]:
# Exercício 3 — modificação guiada (altere apenas o batch_size em cada cópia)
from torch.utils.data import Dataset, DataLoader

loader_batch4 = DataLoader(dataset, batch_size=4)
X_lote4, y_lote4 = next(iter(loader_batch4))
print("batch_size=4  -> X_lote4.shape:", X_lote4.shape, "| y_lote4.shape:", y_lote4.shape)

## 5. Construindo a MLP com `nn.Module`

Agora vamos declarar a estrutura da nossa primeira rede neural: a `MLPIris`. Ela recebe as 5 features de `df_features` e produz 3 números de saída (um "placar" para cada classe).


![Diagrama da Rede](https://drive.google.com/thumbnail?id=1i__wjq7jTTNBtYyTKkcJYpGJU6_EewLq&sz=w1600)

In [ ]:
import torch.nn as nn

class MLPIris(nn.Module):
    def __init__(self, n_entrada=5, n_escondida=8, n_saida=3):
        super().__init__()
        self.camada_entrada = nn.Linear(n_entrada, n_escondida)
        self.ativacao = nn.ReLU()
        self.camada_saida = nn.Linear(n_escondida, n_saida)

    def forward(self, x):
        x = self.ativacao(self.camada_entrada(x))
        return self.camada_saida(x)

modelo = MLPIris()
print(modelo)


### Exercício 4 (modificação guiada) — arquitetura da MLP e leitura de `print(modelo)`

Este exercício tem **duas partes**, realizadas na mesma célula de código abaixo. na Parte 1 você usa a mesma célula-modelo para gerar uma arquitetura **desacoplada de `df_features`** e praticar a leitura de `print(modelo)` em uma MLP qualquer — não apenas na `MLPIris` desta lição; Na Parte 2 você varia o tamanho da camada escondida da `MLPIris`.

**Parte 1 — ler uma arquitetura diferente.** Ainda na mesma célula, instancie a `MLPIris` mais uma vez, agora alterando `n_entrada`, `n_escondida` e `n_saida`: `MLPIris(n_entrada=4, n_escondida=8, n_saida=3)`. Atenção: essa instância **não** serve para `X_tensor` (que tem 5 features) — ela existe apenas para você ler a arquitetura impressa, exatamente como faria com a rede de outra pessoa.

**Parte 2 — modificar a quantidade de neurônios.** Altere apenas o número de neurônios da camada escondida (`n_escondida`) em cada cópia: `4`, `16` e `64`. Em cada cópia, instancie o modelo e execute `print(modelo)`.

**Responda em texto:**

**As perguntas abaixo devem ser respondidas com base em modelo_leitura**
1. Quantos neurônios existem na camada escondida dessa rede?
2. Quantas classes ela pode prever?
3. Qual é a entrada e a saída da primeira camada?
4. A `ReLU` altera o número de elementos do tensor que passa por ela?

**A pergunta abaixo deve ser respondida com base em modelo_escondida**

5. Compare as três arquiteturas impressas com `n_escondida` igual a `4`, `16` e `64`. O que muda em `camada_entrada` (`in_features`/`out_features`) e em `camada_saida` (`in_features`/`out_features`) entre as três? O que permanece fixo no `in_features` de `camada_entrada` e no `out_features` de `camada_saida`, independentemente do valor de `n_escondida` — e por quê?

*Sua resposta aqui:*


In [ ]:
# Exercício 4 — modificação guiada

import torch.nn as nn

class MLPIris(nn.Module):
    # Parte 1 — preencha n_entrada, n_escondida n_saida para ler uma arquitetura fora do padrão de df_features
    def __init__(self, n_entrada=, n_escondida=, n_saida=):
        super().__init__()
        self.camada_entrada = nn.Linear(n_entrada, n_escondida)
        self.ativacao = nn.ReLU()
        self.camada_saida = nn.Linear(n_escondida, n_saida)

    def forward(self, x):
        x = self.ativacao(self.camada_entrada(x))
        return self.camada_saida(x)

modelo_leitura = MLPIris()
print(modelo_leitura)

# Parte 2 — preencha apenas n_escondida em cada instância (4, 16 e 64)

class MLPIris(nn.Module):
    def __init__(self, n_entrada=5, n_escondida=, n_saida=3):
        super().__init__()
        self.camada_entrada = nn.Linear(n_entrada, n_escondida)
        self.ativacao = nn.ReLU()
        self.camada_saida = nn.Linear(n_escondida, n_saida)

    def forward(self, x):
        x = self.ativacao(self.camada_entrada(x))
        return self.camada_saida(x)

modelo4 = MLPIris()
print(modelo4)

class MLPIris(nn.Module):
    def __init__(self, n_entrada=5, n_escondida=, n_saida=3):
        super().__init__()
        self.camada_entrada = nn.Linear(n_entrada, n_escondida)
        self.ativacao = nn.ReLU()
        self.camada_saida = nn.Linear(n_escondida, n_saida)

    def forward(self, x):
        x = self.ativacao(self.camada_entrada(x))
        return self.camada_saida(x)

modelo16 = MLPIris()
print(modelo16)

class MLPIris(nn.Module):
    def __init__(self, n_entrada=5, n_escondida=, n_saida=3):
        super().__init__()
        self.camada_entrada = nn.Linear(n_entrada, n_escondida)
        self.ativacao = nn.ReLU()
        self.camada_saida = nn.Linear(n_escondida, n_saida)

    def forward(self, x):
        x = self.ativacao(self.camada_entrada(x))
        return self.camada_saida(x)

modelo64 = MLPIris()
print(modelo64)



## 6. Predição antes do treinamento

Vamos chamar a MLP recém-criada (com pesos ainda aleatórios) sobre as cinco primeiras amostras de `X_tensor`, e comparar a saída com as classes reais em `y_tensor`.


In [ ]:
saida_inicial = modelo(X_tensor[:5])
print("Saída bruta da MLP (não treinada):")
print(saida_inicial)
print()
print("Classes reais correspondentes (y_tensor[:5]):")
print(y_tensor[:5])


Repare que os três números de cada linha da saída não têm relação alguma com a classe real da amostra correspondente — bem diferente do que aconteceria com a `LogisticRegression` já treinada na M03-L06. Isso acontece porque os pesos de `nn.Linear` nascem aleatórios: essa predição é apenas o ponto de partida. O ajuste desses pesos é o assunto inteiro da M04-L02.

Esse mesmo processo de ajuste, se levado longe demais ou de menos, pode levar o modelo a "aprender demais" os dados de treino (overfitting) ou "de menos" o padrão geral (underfitting) — algo que vamos diagnosticar na prática na próxima lição.

## 7. Um erro comum (proposital)

A célula abaixo tenta chamar a MLP diretamente sobre `df_features` (um `pandas.DataFrame`), sem convertê-lo em tensor. Execute a célula e observe a mensagem de erro real do PyTorch.


In [ ]:
# Célula com erro proposital — execute e observe a mensagem de erro
modelo(df_features)


### Exercício de correção de erro (texto)

**Responda em texto:** por que esse erro ocorre? O que precisaria mudar no código para corrigi-lo? (Não reescreva a célula — apenas explique.)

*Sua resposta aqui:*


## 8. Exercício final — diagnóstico dirigido (código + texto)

Nesta etapa você não vai recontar o que foi feito — vai gerar uma evidência nova, com uma pequena modificação de código, e interpretar essa evidência por escrito comparando com a MLP da Seção 6 Predição antes do treinamento.

**Parte 1 — código.** Execute a célula abaixo (mesmo padrão dos Exercícios 2 a 4). Ela cria uma terceira versão da MLP (`n_escondida=32`) e uma nova predição não treinada — agora sobre **10 amostras** (`X_tensor[:10]`), em vez das 5 amostras usadas na Seção 6.

**Parte 2 — texto.** Com base na saída real que **você acabou de gerar**, responda às três perguntas abaixo. Respostas que apenas recontam os passos da lição, sem citar os valores/shapes que você observou, não serão aceitas.

**Lembre de comparar os valores deste exercício com os valores da seção 6 Predição antes do treinamento**

a) Qual é o novo formato (`.shape`) de `saida_final`? O que determinou cada uma de suas duas dimensões — a de linhas e a de colunas?

b) Os valores numéricos de `saida_final` são diferentes dos de `saida_inicial`, mesmo nas 5 primeiras linhas em comum entre as duas saídas? Por que essa mudança ocorre mesmo reaproveitando o mesmo `X_tensor`? Aponte explicitamente em qual dos cinco elementos a seguir mora essa diferença: tensor, `Dataset`, `DataLoader`, `nn.Module`, pesos aleatórios.

c) Aumentar `n_escondida` de 8 para 32 **e** passar mais amostras (de 5 para 10) aumentou o valor preditivo do modelo? Justifique com base no estado dos pesos (treinado vs. aleatório) — não no tamanho da rede nem na quantidade de dados passados.

*Sua resposta aqui:*


In [ ]:
# Exercício final — modificação guiada (altere apenas n_escondida e o fatiamento de X_tensor abaixo)
modelo_final = MLPIris(n_escondida=32)
saida_final = modelo_final(X_tensor[:10])
print(saida_final.shape)
print(saida_final)


## 9. Desafio extra (opcional)

Para participantes com mais experiência: esta é a única seção da lição em que é permitido escrever código PyTorch **do zero**.

Crie uma nova classe `MLPIrisProfunda(nn.Module)` com **duas** camadas escondidas (por exemplo: `nn.Linear(5, N1)` → ativação → `nn.Linear(N1, N2)` → ativação → `nn.Linear(N2, 3)`), reaproveitando `X_tensor`/`y_tensor` já criados nesta lição. Instancie o novo modelo e execute a mesma predição não treinada sobre `X_tensor[:5]`.

![Diagrama da Rede](https://drive.google.com/thumbnail?id=15j0DH7uxUi6qqq627tMa1UOrIb8zMlNU&sz=w1600)


**Responda em texto:** como o formato da saída se compara ao da `MLPIris` original de uma única camada escondida?

In [ ]:
# Desafio extra — código do zero permitido apenas nesta célula
class MLPIrisProfunda(nn.Module):
    def __init__(self, n_entrada=5, n1=8, n2=4, n_saida=3):
        super().__init__()
        # escreva aqui as camadas nn.Linear e a ativação

    def forward(self, x):
        # escreva aqui o caminho dos dados pelas camadas
        pass



*Sua resposta em texto sobre o desafio extra aqui (se optou por fazê-lo):*

## 10. Reflexão final

Você construiu, nesta lição, toda a estrutura de uma rede neural — tensores, `Dataset`, `DataLoader`, a classe `MLPIris` — e viu que ela já pode ser chamada, mesmo sem nenhum treinamento.

**Pergunta para levar para a M04-L02:** o que precisa acontecer para que essa predição, hoje aleatória, passe a fazer sentido?

*Sua reflexão aqui:*


**Salve os seus dados que foram convertido em tensores**

In [ ]:
import os
import torch
from google.colab import drive

# 1. Conecta o Google Drive do aluno
drive.mount('/content/drive')

# 2. Define o caminho da pasta com os dados do módulo 04
pasta_drive = '/content/drive/MyDrive/dados_modulo_04'

# 3. Define o caminho do novo arquivo de tensores (.pt)
caminho_novo_arquivo = os.path.join(pasta_drive, 'df_features_tensores.pt')

# 4. Salva o seu tensor
torch.save(df_features, caminho_novo_arquivo)

print(f"Novo arquivo salvo com sucesso no Drive: {caminho_novo_arquivo}")


## 11. Checklist final

- [ ] Abri o notebook no Google Colab e salvei uma cópia no meu Google Drive.
- [ ] Executei todas as células de texto e código, na ordem, incluindo a retomada condensada de todo o M03.
- [ ] Reconstruí `df_features`, salvei uma copia no meu drive e registrei, por escrito, se a tabela permanece balanceada entre as três classes.
- [ ] Converti `df_features` em `X_tensor`/`y_tensor` e realizei a modificação guiada de `dtype`, comparando `.dtype` e consumo de memória entre `X_tensor` e `X_tensor_float64`.
- [ ] Criei o `Dataset` e o `DataLoader` e realizei a modificação guiada de `batch_size` para pelo menos dois valores diferentes (4 e 16), comparando o `.shape` dos lotes resultantes.
- [ ] Construí a MLP com `nn.Module` e realizei a Parte 1 do Exercício 4 (modificação guiada do número de neurônios escondidos para três valores diferentes: 4, 16 e 64).
- [ ] Realizei a Parte 2 do Exercício 4 (`MLPIris(n_entrada=4, n_escondida=8, n_saida=3)`) e respondi às perguntas de leitura de arquitetura, identificando `in_features`, `out_features` de cada camada e o efeito da `ReLU` sobre o número de elementos do tensor.
- [ ] Executei a predição não treinada e registrei, por escrito, por que ela ainda não tem valor preditivo.
- [ ] Diagnostiquei por escrito o erro proposital de chamar o modelo diretamente sobre o `DataFrame`.
- [ ] Executei a célula do exercício final (`modelo_final`, `n_escondida=32`, 10 amostras) e respondi às três perguntas de diagnóstico (a, b, c) com base no shape, nos valores e no estado dos pesos observados — não em uma descrição narrativa dos passos.
- [ ] Li o desafio extra (ou completei o desafio, se optei por fazê-lo).
- [ ] Salvei df_features_tensores.pt no meu drive.
- [ ] Preenchi a reflexão final antecipando a pergunta da M04-L02.
- [ ] Finalizei o laboratório da lição M04-L01.


## 12. Orientação de entrega

Salve este notebook com todas as células executadas, na ordem, e com todas as respostas em texto preenchidas. Mantenha a cópia no seu Google Drive — ela será a base direta da M04-L02, que reaproveita `X_tensor`, `y_tensor`, o `Dataset`/`DataLoader` e a classe `MLPIris` construídos aqui, sem reconstrução do zero.
